# 02 — Feature Engineering & Frequency Alignment

Builds the feature matrices and train/val/test splits for all models, organised into
explicit **ablation groups**. Emits **two frames**: the daily matrix (§1–7) and a shared
**weekly W-FRI frame** (§8) that the weekly return-model notebooks read instead of
re-aggregating the daily splits themselves. The weekly frame folds in the leak-corrected
monthly macro lags (previously `02d_macro_features_weekly.ipynb`) and the directional
technical indicators (selected in `02c`).

In [1]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120

## 1. Load raw data

In [2]:
prices = pd.read_csv('../data/raw/daily_prices.csv', index_col=0, parse_dates=True)
fred_d = pd.read_csv('../data/raw/daily_fred.csv',   index_col=0, parse_dates=True)

prices.index = prices.index.tz_localize(None)   # strip tz if present
fred_d.index = fred_d.index.tz_localize(None)

# Align FRED daily onto the yfinance trading-day grid. ffill first (fred_d's index is the
# union of 3 calendars — DFII10/T10YIE on business days + ICSA on Saturdays — so each row
# has NaN for the series whose calendar doesn't include it), THEN reindex with
# method='ffill' for any missing target dates. (Full rationale: 01_eda §1.)
prices = prices.join(fred_d.ffill().reindex(prices.index, method='ffill'))
print('Daily grid:', prices.shape)

Daily grid: (2854, 10)


## 2. Daily features

All transforms are model-ready and observable same-day (weekly notebooks re-aggregate to
W-FRI and add their own 1-week lag). Change conventions match `01_eda` §6/§7: prices →
log-return, rates → Δ (yield change), counts → Δlog.

In [3]:
df = pd.DataFrame(index=prices.index)

# ── Target ───────────────────────────────────────────────────────────────────
df['silver']        = prices['silver']                  # level — kept for tech features (02c)
df['silver_return'] = np.log(prices['silver']).diff()   # log-return (weekly notebooks W-FRI-sum)

# ── YF_DAILY: cross-asset log-returns ─────────────────────────────────────────
for a in ['gold', 'copper', 'usd_index', 'sp500', 'vix', 'oil']:
    df['usd_return' if a == 'usd_index' else f'{a}_return'] = np.log(prices[a]).diff()

# ── GS: gold/silver ratio z-score (train-period stats only — no look-ahead) ───
# The ratio hovers ~80 across the sample, so a fixed pre-2022 baseline captures
# "stretched vs the long-run norm" better than a rolling window (which drifts during
# long deviations like Mar 2020). Train = pre-2022; same mean/std applied to val + test.
_gs       = prices['gold'] / prices['silver']
_gs_train = _gs[_gs.index < '2022-01-01']
df['gs_ratio_z'] = (_gs - _gs_train.mean()) / _gs_train.std()
print(f'gs_ratio_z: train mean={_gs_train.mean():.2f}, std={_gs_train.std():.2f}')

# ── FRED_DAILY: rates as Δyield, jobless claims as Δlog ───────────────────────
df['real_rates_chg'] = prices['real_rates_10y'].diff()
df['breakeven_chg']  = prices['breakeven_10y'].diff()
df['jobless_chg']    = np.log(prices['jobless_claims']).diff()

# ── AR (own-history / weak-form): silver-return lags ──────────────────────────
for lag in [1, 2, 3]:
    df[f'silver_lag{lag}'] = df['silver_return'].shift(lag)

print(df.shape)
df.head()

gs_ratio_z: train mean=78.50, std=8.89
(2854, 15)


/Users/asier.ugartechegmail.com/miniforge3/envs/tf/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,silver,silver_return,gold_return,copper_return,usd_return,sp500_return,vix_return,oil_return,gs_ratio_z,real_rates_chg,breakeven_chg,jobless_chg,silver_lag1,silver_lag2,silver_lag3
Date,,,,,,,,,,,,,,,
2015-01-02,15.734000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.350826,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,16.179001,0.027890,0.014980,-0.016159,0.003288,-0.018447,0.113088,-0.051603,-0.459633,-0.01,-0.07,NaN,NaN,NaN,NaN
2015-01-06,16.603001,0.025869,0.012711,0.003931,0.001312,-0.008933,0.058496,-0.043081,-0.569098,0.01,-0.08,0.0,0.027890,NaN,NaN
2015-01-07,16.510000,-0.005617,-0.007161,-0.002857,0.004253,0.011563,-0.089597,0.014910,-0.581845,-0.02,0.01,0.0,0.025869,0.027890,NaN
2015-01-08,16.351000,-0.009677,-0.001819,0.003926,0.005210,0.017730,-0.126822,0.002874,-0.516747,0.02,0.05,0.0,-0.005617,0.025869,0.02789


## 3. COT positioning — COMEX silver disaggregated report

Weekly Commitments-of-Traders data from the CFTC (`disaggregated_fut`, collected via
`src/collect_cot.py` → `data/raw/cot_silver.csv`). Two net-positioning series:

- `cot_mm_net_pct`   = (Managed Money longs − shorts) / Open Interest — speculative positioning
- `cot_comm_net_pct` = (Producer/Merchant longs − shorts) / Open Interest — hedger positioning

**Lag rigour**: CFTC publishes Friday afternoon with data as-of the preceding Tuesday. The
report dated Friday $t-1$ carries Tuesday $t-1$ data — fully observed before a Friday-$t$
target, so a 1-week lag is leak-safe. Here we forward-fill the release snapshot onto the
daily grid; weekly notebooks add the lag.

In [4]:
cot_path = '../data/raw/cot_silver.csv'
if os.path.exists(cot_path):
    cot = pd.read_csv(cot_path, parse_dates=['report_date']).sort_values('report_date').set_index('report_date')
    mm_net   = cot['M_Money_Positions_Long_All']   - cot['M_Money_Positions_Short_All']
    comm_net = cot['Prod_Merc_Positions_Long_All'] - cot['Prod_Merc_Positions_Short_All']
    oi       = cot['Open_Interest_All']
    cot_feats = pd.DataFrame({'cot_mm_net_pct': mm_net / oi, 'cot_comm_net_pct': comm_net / oi})
    df = df.join(cot_feats.reindex(df.index, method='ffill'))
    print(f'COT merged ({cot.index.min().date()} → {cot.index.max().date()}).')
else:
    print('cot_silver.csv missing — run `python src/collect_cot.py`.')
    df['cot_mm_net_pct'] = np.nan
    df['cot_comm_net_pct'] = np.nan

COT merged (2006-06-13 → 2026-05-05).


## 4. Sentiment (Reddit + news)

Daily FinBERT (news) + Twitter-RoBERTa (Reddit) scores from `03_sentiment.ipynb`:
`reddit_sentiment`, `news_sentiment`, and the combined `sentiment_score`. The auxiliary
count/weight columns are consumed directly from `daily_sentiment.csv` by the volatility
notebook, so they don't enter the split. `SENT` ablates the reddit+news pair;
`sentiment_score` is the combined daily index (kept for the daily LSTM).

In [5]:
sentiment_path = '../data/processed/daily_sentiment.csv'
SENT_COLS = ['reddit_sentiment', 'news_sentiment', 'sentiment_score']
if os.path.exists(sentiment_path):
    sent = pd.read_csv(sentiment_path, index_col=0, parse_dates=True)
    df = df.join(sent[SENT_COLS], how='left')
    print('Sentiment joined:', SENT_COLS)
else:
    print('No sentiment file — run 03_sentiment.ipynb first.')
    for c in SENT_COLS:
        df[c] = np.nan

Sentiment joined: ['reddit_sentiment', 'news_sentiment', 'sentiment_score']


## 5. Feature groups for ablations

Explicit column groups so model notebooks select features by group name rather than
hardcoding lists. Saved as `data/processed/feature_groups.json`. Monthly macro is absent by
design — see `02d_macro_features_weekly.ipynb` for its leak-corrected weekly lags.

In [6]:
FEATURE_GROUPS = {
    'AR':         ['silver_lag1', 'silver_lag2', 'silver_lag3'],
    'GS':         ['gs_ratio_z'],
    'YF_DAILY':   ['gold_return', 'copper_return', 'usd_return', 'sp500_return', 'vix_return', 'oil_return'],
    'FRED_DAILY': ['real_rates_chg', 'breakeven_chg', 'jobless_chg'],
    'COT':        ['cot_mm_net_pct', 'cot_comm_net_pct'],
    'SENT':       ['reddit_sentiment', 'news_sentiment'],
}
missing = [c for g in FEATURE_GROUPS.values() for c in g if c not in df.columns]
print('Missing grouped columns:', missing if missing else 'none')
for g, cols in FEATURE_GROUPS.items():
    print(f'  {g:11s} ({len(cols)}): {cols}')

with open('../data/processed/feature_groups.json', 'w') as f:
    json.dump(FEATURE_GROUPS, f, indent=2)
print('\nSaved -> data/processed/feature_groups.json')

Missing grouped columns: none
  AR          (3): ['silver_lag1', 'silver_lag2', 'silver_lag3']
  GS          (1): ['gs_ratio_z']
  YF_DAILY    (6): ['gold_return', 'copper_return', 'usd_return', 'sp500_return', 'vix_return', 'oil_return']
  FRED_DAILY  (3): ['real_rates_chg', 'breakeven_chg', 'jobless_chg']
  COT         (2): ['cot_mm_net_pct', 'cot_comm_net_pct']
  SENT        (2): ['reddit_sentiment', 'news_sentiment']

Saved -> data/processed/feature_groups.json


## 6. Train / validation / test split
- Train: 2015–2021
- Val:   2022
- Test:  2023–2026 (YTD)

In [7]:
df = df.dropna(subset=['silver_return'])

train = df[df.index < '2022-01-01']
val   = df[(df.index >= '2022-01-01') & (df.index < '2023-01-01')]
test  = df[df.index >= '2023-01-01']

print(f'Train: {train.index[0].date()} → {train.index[-1].date()}  ({len(train)} days)')
print(f'Val:   {val.index[0].date()} → {val.index[-1].date()}  ({len(val)} days)')
print(f'Test:  {test.index[0].date()} → {test.index[-1].date()}  ({len(test)} days)')
print('Columns:', list(df.columns))

Train: 2015-01-05 → 2021-12-31  (1755 days)
Val:   2022-01-03 → 2022-12-30  (251 days)
Test:  2023-01-03 → 2026-05-01  (835 days)
Columns: ['silver', 'silver_return', 'gold_return', 'copper_return', 'usd_return', 'sp500_return', 'vix_return', 'oil_return', 'gs_ratio_z', 'real_rates_chg', 'breakeven_chg', 'jobless_chg', 'silver_lag1', 'silver_lag2', 'silver_lag3', 'cot_mm_net_pct', 'cot_comm_net_pct', 'reddit_sentiment', 'news_sentiment', 'sentiment_score']


## 7. Save feature matrix

In [8]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/features.csv')
train.to_csv('../data/processed/train.csv')
val.to_csv('../data/processed/val.csv')
test.to_csv('../data/processed/test.csv')
print('Saved feature matrix and splits to data/processed/')

Saved feature matrix and splits to data/processed/


## 8. Weekly (W-FRI) feature frame

Single source of truth for the weekly return models. Aggregation matches every notebook's
old `to_weekly()`: returns -> sum, levels (`gs_ratio_z`) / COT -> last, FRED deltas -> sum,
sentiment -> mean. Built **per split then concatenated** so the W-FRI buckets are identical
to the old per-split resample. Base features are **un-lagged** (each model applies its own
1-week lag; the LSTM window supplies it). **TECH** and **MACRO** are **pre-lagged** by
convention, exactly as before. Folds in `02d` (macro) and the directional tech from `02c`.

In [9]:
# ── Base weekly features (per split, then concat → matches old to_weekly bit-for-bit) ──
SUM_W  = ['silver_return', 'gold_return', 'copper_return', 'usd_return',
          'sp500_return', 'vix_return', 'oil_return',
          'real_rates_chg', 'breakeven_chg', 'jobless_chg']
LAST_W = ['gs_ratio_z', 'cot_mm_net_pct', 'cot_comm_net_pct']
MEAN_W = ['reddit_sentiment', 'news_sentiment']

def _weekly_base(split_df):
    parts = []
    for cols, how in [(SUM_W, 'sum'), (LAST_W, 'last'), (MEAN_W, 'mean')]:
        present = [c for c in cols if c in split_df.columns]
        if present:
            parts.append(getattr(split_df[present].resample('W-FRI'), how)())
    return pd.concat(parts, axis=1)

# Guard: split boundaries must fall on Fridays so no W-FRI week straddles two splits.
for _s in (train, val):
    assert _s.index[-1].weekday() == 4, f'split ends {_s.index[-1].date()} (not Friday) — weeks would straddle'

wk = []
for _name, _sdf in [('train', train), ('val', val), ('test', test)]:
    _w = _weekly_base(_sdf)
    _w.insert(0, 'split', _name)
    wk.append(_w)
weekly = pd.concat(wk).sort_index()
weekly.index.name = 'week_end'

# ── Directional technical indicators (02c pool) — full weekly silver, PRE-LAGGED 1w ──
silver_w = prices['silver'].resample('W-FRI').last().dropna()

def _rsi(s, window=14):
    d = s.diff(); g = d.clip(lower=0); l = -d.clip(upper=0)
    ag = g.ewm(span=window, min_periods=window, adjust=False).mean()
    al = l.ewm(span=window, min_periods=window, adjust=False).mean()
    return 100 - 100 / (1 + ag / al.replace(0, np.nan))

_ml  = silver_w.ewm(span=12, adjust=False).mean() - silver_w.ewm(span=26, adjust=False).mean()
_lr  = np.log(silver_w / silver_w.shift(1))
_mx, _mn = silver_w.rolling(52).max(), silver_w.rolling(52).min()
_ma, _sd = silver_w.rolling(20).mean(), silver_w.rolling(20).std()

tech = pd.DataFrame({
    'macd_line':    _ml,
    'macd_hist':    _ml - _ml.ewm(span=9, adjust=False).mean(),
    'rsi_14w':      _rsi(silver_w, 14),
    'mom_5w':       _lr.rolling(5).sum(),
    'roc_13w':      silver_w.pct_change(13),
    'roc_26w':      silver_w.pct_change(26),
    'roc_52w':      silver_w.pct_change(52),
    'price_ma13w':  silver_w / silver_w.rolling(13).mean() - 1,
    'donchian_52w': 2 * (silver_w - _mn) / (_mx - _mn) - 1,
    'bb_pct_b':     (silver_w - (_ma - 2 * _sd)) / (4 * _sd),
}).shift(1)                       # pre-lagged, as the notebooks did in-line
TECH_COLS = list(tech.columns)
weekly = weekly.join(tech.reindex(weekly.index))

# ── Monthly macro: leak-corrected weekly lags (folds in 02d) ──
macro = pd.read_csv('../data/raw/monthly_macro.csv', index_col=0, parse_dates=True)
MACRO_VARS   = [c for c in ['cpi', 'fed_funds', 'ind_prod', 'real_rates'] if c in macro.columns]
N_MACRO_LAGS = 3
MACRO_AVAIL_LAG = {'cpi': 46, 'ind_prod': 48, 'fed_funds': 35, 'real_rates': 46}

def _weekly_macro_lags(weekly_dates, series, lag_days, n_lags=N_MACRO_LAGS):
    s = series.dropna().sort_index()
    avail = s.index + pd.Timedelta(days=lag_days)
    out = np.full((len(weekly_dates), n_lags), np.nan)
    for i, d in enumerate(weekly_dates):
        past = s.values[avail < (d - pd.Timedelta(days=6))]   # observable by F_{t-1}
        if len(past) >= n_lags:
            out[i] = past[-n_lags:][::-1]                      # mlag1 = most recent
    return out

_mac = []
for v in MACRO_VARS:
    _mat = _weekly_macro_lags(weekly.index, macro[v], MACRO_AVAIL_LAG[v])
    _mac.append(pd.DataFrame(_mat, index=weekly.index,
                             columns=[f'{v}_mlag{k}' for k in range(1, N_MACRO_LAGS + 1)]))
macro_w    = pd.concat(_mac, axis=1)
MACRO_COLS = list(macro_w.columns)
weekly = weekly.join(macro_w)

# ── Register the new groups and persist ──
FEATURE_GROUPS['TECH']  = TECH_COLS
FEATURE_GROUPS['MACRO'] = MACRO_COLS
with open('../data/processed/feature_groups.json', 'w') as f:
    json.dump(FEATURE_GROUPS, f, indent=2)

weekly.to_csv('../data/processed/features_weekly.csv')
macro_w.to_csv('../data/processed/macro_weekly_lags.csv')   # kept for 02d back-compat during migration
print('features_weekly.csv:', weekly.shape, '| splits:', weekly['split'].value_counts().reindex(['train','val','test']).to_dict())
print('TECH :', TECH_COLS)
print('MACRO:', MACRO_COLS)

features_weekly.csv: (591, 35) | splits: {'train': 365, 'val': 52, 'test': 174}
TECH : ['macd_line', 'macd_hist', 'rsi_14w', 'mom_5w', 'roc_13w', 'roc_26w', 'roc_52w', 'price_ma13w', 'donchian_52w', 'bb_pct_b']
MACRO: ['cpi_mlag1', 'cpi_mlag2', 'cpi_mlag3', 'fed_funds_mlag1', 'fed_funds_mlag2', 'fed_funds_mlag3', 'ind_prod_mlag1', 'ind_prod_mlag2', 'ind_prod_mlag3']


In [10]:
# Validation: weekly BASE features reproduce the old per-split to_weekly() exactly.
# (TECH is deliberately the new directional set, so it is excluded from this equality check.)
_chk = ['silver_return', 'gold_return', 'usd_return', 'copper_return',
        'sp500_return', 'vix_return', 'oil_return', 'gs_ratio_z']
_agg = {c: ('last' if c == 'gs_ratio_z' else 'sum') for c in _chk}
_old = pd.concat([s[_chk].resample('W-FRI').agg(_agg).dropna()
                  for s in (train, val, test)]).sort_index()
_new = weekly[_chk].dropna()
_common = _old.index.intersection(_new.index)
_maxdiff = (_old.loc[_common] - _new.loc[_common]).abs().to_numpy().max()
print(f'base-feature reproduction: max|old-new| = {_maxdiff:.2e} over '
      f'{len(_common)} weeks x {len(_chk)} cols')
assert len(_common) == len(_old) == len(_new), 'weekly index mismatch vs old to_weekly'
assert _maxdiff < 1e-9, 'weekly base features diverge from the old to_weekly()'
print('Weekly-frame reproduction check PASSED.')

base-feature reproduction: max|old-new| = 0.00e+00 over 591 weeks x 8 cols
Weekly-frame reproduction check PASSED.
